In [0]:
df = spark.read.format("delta").load("dbfs:/mnt/busdataapi/bus_data_delta")
df = df.dropna()

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bricksdawidbrejecki.test_baza.bus_gold")

In [0]:
%sql
CREATE OR REPLACE TABLE bricksdawidbrejecki.test_baza.bus_gold_agg
USING DELTA
AS
select hour(Timestamp) as hour_timestamp, count(distinct VehicleNumber) as ile_autobusow 
from bricksdawidbrejecki.test_baza.bus_gold
where date(Timestamp) = current_date()
group by hour(Timestamp)
order by hour(Timestamp)

In [0]:
agg_df = spark.sql("""
  SELECT hour(Timestamp) as hour_timestamp, 
         count(distinct VehicleNumber) as ile_autobusow 
  FROM bricksdawidbrejecki.test_baza.bus_gold
  WHERE date(Timestamp) = current_date()
  GROUP BY hour(Timestamp)
  ORDER BY hour(Timestamp)
""")
server_name = "serverdawidbrejecki.database.windows.net"
database_name = "sqldawidbrejecki"
username = "dawidbrejecki"
password = "Photology18!"
jdbc_url = f"jdbc:sqlserver://{server_name}:1433;database={database_name};user={username};password={password};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30"
agg_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "bus_stats") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .mode("overwrite") \
    .save()